In [ ]:
'''
＜解析の流れ＞

ダミー変数化
テストデータの分離

xとyへの分割
教師データの分割

外れ値の除外
（標準化）
欠損値の補完・削除
特徴量の絞り込み
多項式・交互作用特徴量の追加

訓練データで学習
検証データで評価

テストデータで最終評価
'''

In [ ]:
#import
import pandas as pd

from sklearn.model_selection import train_test_split  #データの分割
from sklearn.covariance import MinCovDet  #マハラノビス距離

from sklearn.tree import DecisionTreeClassifier  #決定木（標準化は不要）

from sklearn.linear_model import Ridge  #リッジ回帰

In [ ]:
#データの前処理
df=pd.read_csv('sukkiri-ml-codes (1)\datafiles\Bank.csv')
df.head(5)


In [ ]:
#ダミー変数化
str_col_name=['job','default','marital','education','housing','loan','contact','month']
str_df = df[str_col_name]  #文字列を抜き出す
str_df2=pd.get_dummies(str_df,drop_first=True)  #複数列を一気にダミー変数化

num_df = df.drop(str_col_name,axis=1)  #数値列を抜き出す
df2 = pd.concat([num_df,str_df2,],axis=1)  #結合


#不要な特徴量の削除
df2=df2.drop('id',axis=1)

データの分割、学習・検証データにおける外れ値の削除

In [ ]:
#x,y列の指定
y_cols=['y']
x_cols=[c for c in df2.columns if c!='y']

#訓練&検証データとテストデータに分割
train_val,test = train_test_split(df2,test_size=0.1,random_state=9)
train_val.head()

In [ ]:
print(df2.columns)
print()
print(df2.dtypes)

In [ ]:
#マハラノビス距離を求める
#train_val全体でやるとLinAlgErrorになったので、今回はダミー変数を除く
mcd2 =MinCovDet(random_state=0,support_fraction=0.7)
num_train_val_name = ['age', 'amount', 'day', 'duration', 'campaign', 'previous', 'y'] #数値列を抜き出す
num_train_val=train_val[num_train_val_name]

mcd2.fit(num_train_val) 
dis = pd.Series(
    mcd2.mahalanobis(num_train_val),
    index=train_val.index
)
dis=pd.Series(dis)

#箱ひげ図を描く
dis.plot(kind="box")

In [ ]:
#外れ値の削除
no=dis[dis>12000].index
train_val2 = train_val.drop(no)

num_train_val2=train_val2[num_train_val_name]
mcd2.fit(num_train_val2) 
dis2 =mcd2.mahalanobis(num_train_val2)
dis2=pd.Series(dis2)
dis2.plot(kind="box")

In [ ]:
#train_val2を学習データとして使用する。

#訓練データと検証データに分割
x_train,x_val,y_train,y_val = train_test_split(train_val2[x_cols],train_val2[y_cols],test_size=0.2,random_state=0)

欠損値補完

In [ ]:
#欠損値補完
print(train_val2.isnull().sum())

In [ ]:
#durationに多くの欠損値があるので補完する

#リッジ回帰でのdurationの予測モデル

#正則化項の定数を0.01~20まで検証
best_ridgescore=0
best_alpha=0
for i in (1,2001):
    ridgeModel=Ridge(alpha=(i/100))
    ridgeModel.fit()
    print(ridgeModel)



決定木

In [ ]:


#学習
best_depth=0
best_score=0
for i in range(1,21):
    model=DecisionTreeClassifier(max_depth=i,random_state=0,class_weight='balanced')  #不均衡データに配慮
    model.fit(x_train,y_train)
    score=model.score(x_val,y_val)
    print(f'決定木の深さ={i}  検証データでのスコア={score}')
    if best_score < score :
        best_depth=i
        best_score=score

print(f'完成したモデルの　決定木の深さ={best_depth}  検証データでのスコア={best_score}')
final_model=DecisionTreeClassifier(max_depth=best_depth,random_state=0,class_weight='balanced')

In [ ]:
#完成したモデルをテストデータで評価
final_model.fit(test[x_cols],test[y_cols])
score2=final_model.score(test[x_cols],test[y_cols])
print(f'テストデータでのスコアは{score2}')